# 05 Visualization

Create a histogram, convergence plot, and simple ZIP-level Folium map.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.monte_carlo import predict_for_zip_hour
from src.validation import convergence_test
from src.visualization import create_zip_probability_map, plot_convergence, plot_simulated_counts

In [ ]:
rate_table = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "crash_rate_table.csv", dtype={"zip_code": str})
row = rate_table.iloc[0]

result = predict_for_zip_hour(
    rate_table,
    zip_code=row["zip_code"],
    day_of_week=row["day_of_week"],
    hour=int(row["hour"]),
    weather_condition="clear",
    random_seed=42,
)

histogram_path = PROJECT_ROOT / "outputs" / "figures" / "notebook_simulated_counts.png"
plot_simulated_counts(result["simulated_counts"], histogram_path)

In [ ]:
convergence_df = convergence_test(
    crash_count=result["crash_count"],
    observed_hours=result["observed_hours"],
    random_seed=42,
)
convergence_path = PROJECT_ROOT / "outputs" / "figures" / "notebook_convergence.png"
plot_convergence(convergence_df, convergence_path)

In [ ]:
cleaned_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "nyc_crashes_clean.csv", dtype={"zip_code": str})
zip_coordinates = (
    cleaned_df.dropna(subset=["latitude", "longitude"])
    .groupby("zip_code", as_index=False)[["latitude", "longitude"]]
    .mean()
)

map_rows = []
for _, rate_row in rate_table.head(20).iterrows():
    sim = predict_for_zip_hour(
        rate_table,
        zip_code=rate_row["zip_code"],
        day_of_week=rate_row["day_of_week"],
        hour=int(rate_row["hour"]),
        num_trials=2000,
        random_seed=42,
    )
    map_rows.append(
        {
            "zip_code": sim["zip_code"],
            "probability_at_least_one": sim["probability_at_least_one"],
        }
    )

results_df = pd.DataFrame(map_rows).merge(zip_coordinates, on="zip_code", how="left")
map_path = PROJECT_ROOT / "outputs" / "maps" / "zip_probability_map.html"
create_zip_probability_map(results_df, map_path)